In [ ]:
import pandas as pd
from tabulate import tabulate
import os
FEATURES = ['epre','pusch_snr','p_ue','ul_mcs','cqi','ul_bitrate',
            'dl_mcs','dl_retx','ul_tx','dl_tx','ul_retx','dl_bitrate','dl_err','ul_err']

def format_feature_stats_csv(path):
    df = pd.read_csv(path)
    df = df[["feature", "normal_mean", "normal_std", "attack_mean", "attack_std"]]
    df.iloc[:, 1:] = df.iloc[:, 1:].round(4)

    # Add "(Mbps)" to bitrate-related features
    df["feature"] = df["feature"].apply(
        lambda x: f"{x} (Mbps)" if x in ["dl_bitrate", "ul_bitrate"] else x
    )

    return tabulate(
        df.values.tolist(),
        headers=["Feature", "Normal Mean", "Normal Std", "Attack Mean", "Attack Std"],
        tablefmt="grid"
    )

def format_matrix_csv(path):
    df = pd.read_csv(path, index_col=0)
    df = df[FEATURES].round(5)
    df.insert(0, "Timestep", df.index)
    return tabulate(df.values.tolist(), headers=["Timestep"]+FEATURES, tablefmt="grid")

def format_csv_table(path, title):
    try:
        df = pd.read_csv(path)
        df = df.round(5)
        return f"**{title}**\n" + tabulate(df.values.tolist(), headers=df.columns, tablefmt="grid")
    except Exception as e:
        return f"[Error reading {path}: {e}]"

def make_prompt(stats_tbl, input_tbl, pred, lime_tbl, shap_tbl, shap_global_tbl):
    label = "Anomalous" if pred==1 else "Normal"
    return f"""
You are a cybersecurity expert analyzing LSTM outputs for DDoS detection. Respond with 3 sections:
Anomaly Summary (3-6 bullet points), Misclassification Likelihood (1-2 bullets), and Mitigation Steps (2-4 bullets).

---

**General Data Distribution (Feature Statistics)**
{stats_tbl}

---

**Input Sequence to LSTM Model** (3 timesteps × 14 features)
{input_tbl}

---

**Model Prediction Output:** `{pred}` (*{label}*)

---

{lime_tbl}

{shap_tbl}

---

{shap_global_tbl}

---

**Task:**
Provide a human-readable summary of the model's decision.
Highlight which features or patterns most influenced the classification outcome.
What is the likelihood that this instance is a misclassification?
Finally, suggest actionable mitigation steps for the network operator.
""".strip()

# ---------- build prompts ----------
# ---------- Build zero-shot prompts for all entries in manifest ----------
man = pd.read_csv("../XAI_python_scripts/prompt_assets/prompts_manifest.csv")
stats_tbl = format_feature_stats_csv("../XAI_python_scripts/prompt_assets/feature_stats_raw.csv")
shap_global_tbl = format_csv_table("../XAI_python_scripts/prompt_assets/shap_global_heatmap.csv", "Global SHAP Feature Importance")

# Ensure output directory exists
os.makedirs("Zero_Shot_Prompts", exist_ok=True)

for _, row in man.sort_values("prompt_id").iterrows():
    pid = int(row["prompt_id"])
    pred = int(row["y_pred"])

    input_tbl = format_matrix_csv(f"../XAI_python_scripts/prompt_assets/input_raw_{pid}.csv")
    lime_tbl  = format_csv_table(f"../XAI_python_scripts/prompt_assets/lime_{pid}.csv",  "LIME Local Explanation")
    shap_tbl  = format_csv_table(f"../XAI_python_scripts/prompt_assets/shap_{pid}.csv",  "SHAP Local Heatmap")

    prompt_txt = make_prompt(stats_tbl, input_tbl, pred, lime_tbl, shap_tbl, shap_global_tbl)

    with open(f"Zero_Shot_Prompts/prompt_zero_shot_{pid}.txt", "w", encoding="utf-8") as f:
        f.write(prompt_txt)

print("✅ Wrote zero-shot prompts for all manifest entries, including IDs 11–14.")



In [ ]:
import os
import pandas as pd
from tabulate import tabulate

# ------------------------------
# Config
# ------------------------------
ASSETS = "../XAI_python_scripts/prompt_assets/"
MANIFEST = f"{ASSETS}/prompts_manifest.csv"
FEATURE_STATS = f"{ASSETS}/feature_stats_raw.csv"
SHAP_GLOBAL = f"{ASSETS}/shap_global_heatmap.csv"

OUT_DIR = "Few_Shot_Prompts"
os.makedirs(OUT_DIR, exist_ok=True)

FEATURES = [
    "epre", "pusch_snr", "p_ue", "ul_mcs", "cqi", "ul_bitrate",
    "dl_mcs", "dl_retx", "ul_tx", "dl_tx", "ul_retx", "dl_bitrate", "dl_err", "ul_err"
]

# ------------------------------
# Helpers
# ------------------------------
def tbl_feature_stats(path):
    df = pd.read_csv(path)
    df = df[["feature", "normal_mean", "normal_std", "attack_mean", "attack_std"]]
    df.iloc[:, 1:] = df.iloc[:, 1:].round(4)
    return tabulate(
        df.values.tolist(),
        headers=["Feature", "Normal Mean", "Normal Std", "Attack Mean", "Attack Std"],
        tablefmt="grid"
    )

def tbl_matrix(path):
    df = pd.read_csv(path, index_col=0)
    df = df[FEATURES].round(5)
    df.insert(0, "Timestep", df.index)
    return tabulate(df.values.tolist(), headers=["Timestep"] + FEATURES, tablefmt="grid")

def tbl_csv(path, title):
    try:
        df = pd.read_csv(path).round(5)
        return f"**{title}**\n" + tabulate(df.values.tolist(), headers=df.columns, tablefmt="grid")
    except Exception as e:
        return f"[Error reading {path}: {e}]"

def input_block(pid, pred, stats_tbl, shap_global_tbl):
    label = "Anomalous" if pred == 1 else "Normal"
    input_tbl = tbl_matrix(f"{ASSETS}/input_{pid}.csv")
    lime_tbl = tbl_csv(f"{ASSETS}/lime_{pid}.csv", "LIME Local Explanation")
    shap_tbl = tbl_csv(f"{ASSETS}/shap_{pid}.csv", "SHAP Local Heatmap")
    return (
        f"**General Data Distribution (Feature Statistics)**\n"
        f"{stats_tbl}\n\n"
        f"**Input Sequence to LSTM Model** (3 timesteps × 14 features)\n"
        f"{input_tbl}\n\n"
        f"**Model Prediction Output:** `{pred}` (*{label}*)\n\n"
        f"{lime_tbl}\n\n"
        f"{shap_tbl}\n\n"
        f"{shap_global_tbl}"
    )

# ------------------------------
# Example Assistant Responses
# ------------------------------
EXAMPLE_13_OUTPUT = """
Anomaly Summary.
Download speed is abnormally low: The downlink bitrate (≈0.004 Mbps) is far below what’s normal, even compared to known attack data.
Very little data moving downstream: The dl_tx values (around 20–21) are tiny compared to typical traffic levels, suggesting the link is barely active.
Radio quality looks healthy: The connection quality indicators (cqi, pusch_snr, and ul_mcs) are solid, and there are virtually no retransmissions or errors.

Misclassification Likelihood.
Moderate chance of a false negative: 
The system looks fine radio-wise but is transferring almost no data,  a pattern that could also occur during a stealthy or partial DDoS disruption.

Mitigation Steps
Apply a safety check after prediction: Add a simple rule to flag any case where downlink bitrate stays under 0.01 Mbps and transmission count under 50 for several timesteps, even if the model says “normal.”
Cross-check with network data: Combine LSTM outputs with core-network indicators, packet rates, tunnel sessions, or server response anomalies, so low-traffic attacks don’t slip through.
"""

EXAMPLE_14_OUTPUT = """
Anomaly Summary
Unusual traffic balance: The uplink throughput is very high (around 17 Mbps) while the downlink rate is extremely low (around 0.09 Mbps).
Good radio conditions: Signal quality metrics (cqi ≈ 15, pusch_snr ≈ 25) look healthy, and there are no noticeable transmission errors or retransmissions. So the issue isn’t with the radio link.
The same behavior repeats consistently across all three timesteps — the device is sending a lot but barely receiving.
The model’s decision was driven mainly by the high uplink bitrate and volume counters, which stand out compared to what it’s seen in normal traffic.
This pattern fits a device or user endpoint sending large volumes of data continuously — possibly a bot participating in an outbound DDoS event.

Misclassification Likelihood
Moderate but credible alert: Even though the radio health looks fine, the strong and persistent imbalance between upload and download activity is unusual.
While it could happen in legitimate cases (like cloud backup or telemetry uploads), the pattern aligns more closely with anomaly or attack-like behavior.

Mitigation Steps
Immediate validation: Check flow logs for that user or IP — look for sustained uplink packet floods, repeated destination IPs, or unusual port activity.
Network controls: If the traffic looks malicious, apply uplink rate limits or access control filters at the GTP or core router level to reduce potential DDoS impact.
Improve model awareness: Add new features that measure traffic symmetry (like UL/DL bitrate ratio) or throughput vs. CQI — these can help the model better distinguish between legitimate uploads and attack traffic.
Training and calibration: Include examples of normal high-uplink workloads (e.g., data syncs or video uploads) during retraining so the model can tell them apart from DDoS-style behavior.
"""

# ------------------------------
# Headers / Task
# ------------------------------
HEADER = (
    "System Prompt: You are a cybersecurity expert analyzing LSTM outputs for DDoS detection.\n"
    "Respond with 3 sections: Anomaly Summary (3–6 bullet points), Misclassification Likelihood (1–2 bullets), and Mitigation Steps (2–4 bullets)."
)

TASK_FOR_LLM = (
    "Task For LLM:\n"
    "Provide a human-readable summary of the model's decision. Highlight which features or patterns most influenced the classification outcome. "
    "Assess the likelihood of misclassification and suggest actionable mitigation steps for the network operator."
)

# ------------------------------
# Build few-shot prompts (first 10 ids)
# ------------------------------
man = pd.read_csv(MANIFEST).sort_values("prompt_id")

stats_tbl = tbl_feature_stats(FEATURE_STATS)
shap_global_tbl = tbl_csv(SHAP_GLOBAL, "Global SHAP Feature Importance")

pred13 = int(man.loc[man["prompt_id"] == 13, "y_pred"].iloc[0]) if (man["prompt_id"] == 13).any() else 0
pred14 = int(man.loc[man["prompt_id"] == 14, "y_pred"].iloc[0]) if (man["prompt_id"] == 14).any() else 1

EX13_INPUT = input_block(13, pred13, stats_tbl, shap_global_tbl)
EX14_INPUT = input_block(14, pred14, stats_tbl, shap_global_tbl)

for pid in range(1, 11):
    row = man.loc[man["prompt_id"] == pid]
    if row.empty:
        print(f"⚠️ Skipping ID {pid}: not found in manifest")
        continue
    pred = int(row["y_pred"].iloc[0])
    NEW_INPUT = input_block(pid, pred, stats_tbl, shap_global_tbl)

    doc = (
        f"{HEADER}\n\n"
        f"User Prompt:\n{EX13_INPUT}\n\n"
        f"Assistant Response:\n{EXAMPLE_13_OUTPUT}\n\n"
        f"User Prompt:\n{EX14_INPUT}\n\n"
        f"Assistant Response:\n{EXAMPLE_14_OUTPUT}\n\n"
        f"=== New Case ===\n"
        f"User Prompt:\n{NEW_INPUT}\n\n"
        f"{TASK_FOR_LLM}"
    )

    with open(os.path.join(OUT_DIR, f"fewshot_id_{pid}.txt"), "w", encoding="utf-8") as f:
        f.write(doc)

print(f"✅ Few-shot files with explicit User/Assistant markers written to {OUT_DIR}")